# DATA Practical 8 - Inferential Statistics Tutorial - Model Answers



## A Data Journey In Inferential Statistics

**Pandas API Reference**: https://pandas.pydata.org/pandas-docs/stable/reference/index.html

**Matplotlib API Reference**: https://matplotlib.org/stable/api/index.html

**Seaborn API Reference**: https://seaborn.pydata.org/api.html

**Scipy Stats**: https://docs.scipy.org/doc/scipy/tutorial/stats.html

**t table Reference**:  http://www.ttable.org

**F table Reference**  https://www.sjsu.edu/faculty/gerstman/StatPrimer/F-table.pdf

**Chi-Square table Reference** https://people.smp.uq.edu.au/YoniNazarathy/stat_models_B_course_spring_07/distributions/chisqtab.pdf


***

## Preamble


**(1)** To get started with today's practical, you can go to Lecture 16 and try to replicate the inferential statistics we saw during the lecture, i.e., independent and paired t test, ANOVA and Chi-Square. This will enable you to understand better how these tests operate under the hood.

**(2)** Spend a few minutes to check:
* The statistical tests that may be useful in this practical https://docs.scipy.org/doc/scipy/reference/stats.html#statistical-tests (Hint: check in particular the part "Hypothesis Tests and related functions" and the short description next to the function names)

* The functions of the t distribution https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html?highlight=stats%20t%20ppf (Hint: they are very similar to those we used last week for the normal distribution)

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math as m
import pandas as pd

## One Sample t test

A recent report by the Council of York shows an average wage of £11.71/h. Cucina Inc. is a company providing culinary services in York. Cucina wants to make sure its employees are paid similarly to the local average wage. Cucina must decide if a raise is needed (to keep employees happy). 

In [3]:
cucina = [12.65, 12.67, 11.9, 10.45, 13.5, 12.95, 11.77]
mu = 11.71

**T1) Calculate the sample standard deviation analytically (implement the function). Then, use the approppriate Numpy method to check your results**

In [4]:
#Calculate mean
m = np.mean(cucina)


sampleStd = 0
for c in cucina:
    sampleStd += np.power(m-c, 2)
sampleStd = np.sqrt(sampleStd / (len(cucina)-1))

print("Sample standard deviation: ", sampleStd)

print("Sample standard deviation using Numpy: ", np.std(cucina, ddof=1))

Sample standard deviation:  0.9982484660644364
Sample standard deviation using Numpy:  0.9982484660644364


**T2) State the null and alternative hypotheses**

* H0: Cucina Inc. employees earn the same as others: μ = 11.71
* H1: Cucina Inc. employees wage is significantly **different** to the mean wage μ != 11.71   **(two-tailed test)**

This could also be considered as a one-tailed test i.e., it only makes sense to raise the wage if Cucina Inc. employees are underpaid (μ < 11.71). This is also fine.

**T3) Perform the appropriate t test to evaluate the hypotheses for significance level $\alpha = 0.05$. You must show your calculations, i.e., calculate the result analytically, but you may use the appropriate scipy.stats method to check your results.**

In [5]:
cucina = [12.65, 12.67, 11.9, 10.45, 13.5, 12.95, 11.77]
mu = 11.71

#Using t formula
t1 = (np.mean(cucina) - mu)/(np.std(cucina, ddof=1)/np.sqrt(len(cucina)))

#Using library
tStat, pValue = stats.ttest_1samp(cucina, mu, alternative='two-sided')

print("Analytically:", t1)
print("tStat:", tStat, ", pValue:", pValue)

Analytically: 1.484220396588648
tStat: 1.4842203965886478 , pValue: 0.18828316438230835


**T4) Calculate the critical value for the given significance level $\alpha$**

**Hint:** Think of whether this is a one- or two-tailed test and check the stats.t.ppf function

In [6]:
#Critical value one-tailed (not dividing alpha)
alpha = 0.05
dof   = len(cucina)-1

t_crit = stats.t.ppf(alpha/2, df=dof)
print("t Critical value: %.3f" % (np.abs(t_crit)))

t Critical value: 2.447


**T5) Draw appropriate conclusions to accept or reject the null hypothesis**

With significance level α=0.05 (two-tailed):  critical value = 2.447

The t statistic result = 1.484 < 2.447

* We cannot reject H0

* There is insufficient evidence that Cucina employees wage  is significantly different than the average

In [7]:
#Using the t value
print("t(%d)=%.3f < %.3f=t_crit -> DO NOT REJECT H0 (t statistic does not exceed the critical value) " %
      (dof, np.abs(tStat), np.abs(t_crit)))

#Using the p value and alpha significance level
print("p=%.5f > %.3f=α -> H0 IS NOT REJECTED (higher than α) " %(pValue/2, alpha))

t(6)=1.484 < 2.447=t_crit -> DO NOT REJECT H0 (t statistic does not exceed the critical value) 
p=0.09414 > 0.050=α -> H0 IS NOT REJECTED (higher than α) 


**Note** 

* In the example above, when we use the p-value, scipy.stats by default gives the probability for a two-tailed test. 
* The p-value for one-tailed tests can be backed out from the two-tailed tests. 
* With symmetric distributions, the p-value for one-tailed tests is just half of the two-tailed pvalue

***
***

## Two Samples Independent t test

**A/B Testing @ Netflix**

“Displaying better artwork will result in greater engagement and retention by helping members discover stories they will enjoy faster”

In [8]:
#Currently used artwork
con = np.array([0.5, 3.3, 6.7, 11.3, 5.4, 0, 0.3, 1, 2, 10.6, 10.2, 4.6, 12.2, 9.3, 11.6, 2.8, 7.1, 6.2, 12.3, 1.8, 0.2, 3.7, 9.2, 15.9, 8.3, 10.2])

#Candidate new artwork
art = np.array([11.4, 9, 11, 3.9, 5.5, 7.7, 13.4, 10.7, 13.6, 7.4, 2.1, 4.9, 21.5, 5.3, 3.8, 9.4, 13.1])

**T6) State the null and alternative hypotheses**

**Hypotheses**

* H0: Better artwork does not result in greater member engagement μD = μArt- μCon = 0  -> μArt = μCon

* H1: Better artwork does lead to significantly different member engagement  μArt > μCon



**T7) Perform the appropriate t test to evaluate the hypotheses for significance level $\alpha = 0.05$. You must show your calculations, i.e., calculate the result analytically, but you may use the appropriate scipy.stats method to check your results.**

In [9]:
dof = len(con) + len(art) - 2
print("DoF:", dof)

sp  =( (len(art)-1) * np.power(np.std(art, ddof=1),2) + (len(con)-1) * np.power(np.std(con, ddof=1),2) ) / (len(con) + len(art) - 2)
print("SP:", sp)

t = (np.mean(art) - np.mean(con)) / np.sqrt( (sp/len(art)) + (sp/len(con)) )

tStat, pValue = stats.ttest_ind(art, con)

print("Analytically:", t)
print("tStat:", tStat, ", pValue:", pValue)

DoF: 41
SP: 22.09677353492992
Analytically: 1.7935291639609987
tStat: 1.7935291639609987 , pValue: 0.08026646494143931


**T8) Calculate the critical value for the given significance level $\alpha$**

**Hint:** Think of whether this is a one- or two-tailed test

In [9]:
#Critical value one-tailed (alpha is not split)
alpha = 0.05

t_crit = stats.t.ppf(1-alpha, df=dof)
print("t Critical value: %.3f" % (t_crit))

t Critical value: 1.683


**T9) Draw appropriate conclusions**

With significance level α=0.05, df=41 (one-tailed):  critical value = 1.683

The t statistic result is 1.7935 > 1.683 (at the edge of the distribution)

* We can reject H0 

* There is statistically significant evidence that displaying better artwork results in different member engagement.


In [10]:
#Using the t value
print("t(%d)=%.3f > %.3f=t_crit -> REJECT H0 (t statistic exceeds the critical value) " %
      (dof, np.abs(tStat), np.abs(t_crit)))

#Using the p value and alpha significance level
print("p=%.4f < %.3f=α -> REJECT H0 (lower than α) " %(pValue/2, alpha))

t(41)=1.794 > 1.683=t_crit -> REJECT H0 (t statistic exceeds the critical value) 
p=0.0401 < 0.050=α -> REJECT H0 (lower than α) 


***
***

## Two Samples Related t test

**A/B Testing @ Netflix**

“Displaying better artwork will result in greater engagement and retention by helping members discover stories they will enjoy faster”

Does showing better artwork (i.e., the alternative artwork) lead to increased member engagement when the artworks are shown to the same members?

In [11]:
#Assume that the first 10 entries correspond to engagement time from the same members in the two arrays
con10 = con[:10]
art10 = art[:10]

**T10) State the null and alternative hypotheses**

**Hypotheses**

* H0: Better artwork does not result in greater member engagement μD = μArt- μCon = 0  -> μArt = μCon

* H1: Better artwork does lead to significantly increased member engagement  μArt > μCon




**T11) Perform the appropriate t test to evaluate the hypotheses for significance level $\alpha = 0.05$. You must show your calculations, i.e., calculate the result analytically, but you may use the appropriate scipy.stats method to check your results.**

In [12]:
#using the formula
D = art10 - con10
t = np.mean(D)/(np.std(D, ddof=1)/np.sqrt(len(D)))

#using the statistic 
tStat, pValue = stats.ttest_rel(art10, con10)


print("Analytically:", t)
print("tStat:", tStat, ", pValue:", pValue)

Analytically: 2.4339749963061945
tStat: 2.4339749963061945 , pValue: 0.037735725308880724


**T12) Calculate the critical value for the given significance level $\alpha$**

**Hint:** Think of whether this is a one- or two-tailed test

In [13]:
#Critical value one-tailed (on the positive edge)
alpha=0.05

dof=len(D)-1

t_crit = stats.t.ppf(1-alpha, df=dof)

print("t Critical value: %.3f" % (t_crit))

t Critical value: 1.833


**T13) Draw appropriate conclusions**

With significance level α=0.05, df=9 (one-tailed): critical value = 1.83311

The t statistic result is 2.434 > 1.83311

* We can reject H0, 
* There is sufficient statistically significant evidence between the means of the  populations from which our observations were drawn 
* Better artwork increases member engagement


In [14]:
#Using the t value
print("t(%d)=%.3f > %.3f=t_crit -> REJECT H0 (t statistic exceeds the critical value) " %
      (dof, np.abs(tStat), np.abs(t_crit)))

#Using the p value and alpha significance level
#According to the Scipy documentation, the pvalue is for a two-tailed t-test, 
# so we must divide the p by 2 for one-tailed tests
#This is the easiest way to convert a two-tailed test into a one-tailed test
print("p=%.6f < %.3f=α -> REJECT H0 (lower than α) " %(pValue/2, alpha))

t(9)=2.434 > 1.833=t_crit -> REJECT H0 (t statistic exceeds the critical value) 
p=0.018868 < 0.050=α -> REJECT H0 (lower than α) 
